# 🚀 RAG Email Assistant - Launcher

This notebook provides three modes:
- **Quick Start**: Load existing index and launch demo
- **Full Pipeline**: Build index from scratch
- **Fine-tune**: Train custom embedding model

## 1. Environment Setup

In [ ]:
# Install dependencies in Colab/remote environment
!pip install -q sentence-transformers faiss-cpu gradio pyyaml openai tqdm

# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version}")

## 2. Mode Selection

In [ ]:
# Configuration
MODE = "quick"  # Options: "quick", "full", "finetune"
DATASET = "hospital"  # Options: "hospital", "corruption"

print(f"Mode: {MODE}")
print(f"Dataset: {DATASET}")

## 3. Quick Start Mode

Load existing index and launch demo directly.

In [ ]:
if MODE == "quick":
    from config.config import load_config
    from retrieval.retriever import HybridRetriever
    from retrieval.reranker import CrossEncoderReranker
    from generation.rag_generator import RAGGenerator
    from app.gradio_app import create_demo
    
    # Load configuration
    config = load_config(f'{DATASET}_base', project_root=PROJECT_ROOT)
    
    # Check if index exists
    if not config.index_path.exists():
        print(f"❌ Index not found: {config.index_path}")
        print("Please run 'Full Pipeline' mode first to build the index.")
    else:
        print(f"✅ Index found: {config.index_path}")
        print("\n🔄 Loading components...")
        
        # Initialize retriever (automatically loads index)
        retriever = HybridRetriever(config)
        
        # Initialize reranker
        reranker = CrossEncoderReranker(config.reranker_model)
        
        # Initialize generator
        generator = RAGGenerator(config)
        
        print("\n🎨 Launching Gradio demo...")
        
        # Create and launch demo
        demo = create_demo(retriever, reranker, generator, config)
        demo.launch(share=True, server_port=7860)

## 4. Full Pipeline Mode

Build index from scratch and launch demo.

In [ ]:
if MODE == "full":
    from config.config import load_config
    from data.loader import EmailDataLoader
    from retrieval.indexer import FAISSIndexBuilder
    from retrieval.retriever import HybridRetriever
    from retrieval.reranker import CrossEncoderReranker
    from generation.rag_generator import RAGGenerator
    from app.gradio_app import create_demo
    
    # Load configuration
    config = load_config(f'{DATASET}_base', project_root=PROJECT_ROOT)
    
    print("📊 Step 1/4: Loading data...")
    loader = EmailDataLoader(config)
    documents = loader.load_documents()
    print(f"   ✅ Loaded {len(documents)} documents")
    
    print("\n🔨 Step 2/4: Building FAISS index...")
    indexer = FAISSIndexBuilder(config)
    indexer.build_index(documents)
    indexer.save_index()
    print(f"   ✅ Index saved to: {config.index_path}")
    
    print("\n🔍 Step 3/4: Initializing retriever...")
    retriever = HybridRetriever(config)
    reranker = CrossEncoderReranker(config.reranker_model)
    generator = RAGGenerator(config)
    print("   ✅ Components initialized")
    
    print("\n🎨 Step 4/4: Launching Gradio demo...")
    demo = create_demo(retriever, reranker, generator, config)
    demo.launch(share=True, server_port=7860)

## 5. Fine-tune Mode

Train custom embedding model (to be implemented).

In [ ]:
if MODE == "finetune":
    print("🎯 Fine-tuning mode")
    print("TODO: Implement fine-tuning pipeline")
    # Will be implemented in later phase

## 6. Manual Testing (Optional)

Test retrieval and generation manually.

In [ ]:
# Example: Test retrieval
if MODE in ["quick", "full"]:
    test_query = "What did David R. Park request from Katherine?"
    
    print(f"Query: {test_query}\n")
    
    # Retrieve documents
    docs = retriever.retrieve(test_query, top_k=5, use_rerank=True)
    
    print(f"Retrieved {len(docs)} documents:\n")
    for i, doc in enumerate(docs):
        score = doc.get('rerank_score', doc.get('score', 0))
        thread_id = doc['metadata'].get('thread_id')
        print(f"[{i+1}] Thread: {thread_id} | Score: {score:.4f}")
        print(f"    {doc['content'][:200]}...\n")
    
    # Generate answer
    answer = generator.generate(test_query, docs)
    print(f"\n💡 Answer:\n{answer}")